# Пунктуация и заглавные буквы + BERT

## Задача 
Есть идеальная ACR (Audio Character Recognition), которая не умеет расставлять знаки препинания и заглавные буквы. Нужно по текстам без пунктуации и заглавных букв восстановить нормальный текст.

## Решение
### Идейно
Для каждого слова будем проставлять 2 типа меток (2 головы берта): заглавная буква и знак препинания после слова (на обеих головах есть метка ничего не делать). Так как слово может состоять из нескольких токенов, метки нужно выравнивать по subword: для заглавной буквы - первый токен покрывающий слово, для пунктуации - последний. 

Такая задача не решает проблему с абрревиатурами и сокращениями, но для этого существует отдельныая задача - NER (Named Entity Recognition), которую можно решать после восстановления синтаксиса.

### Сбор датасета для обучения
Возьмем датасет из русских текстов и очистим от неприятной пунктуации (я чистил только смайлики). 

Сплитанем по словам и для каждого слова проставим метки.

Выравняем метки по границам слов.

### Архитектура модели

На хвост энкодера из берта повесим две FNN (2 головы)

### Обучение

Все как обычно, один важный момент, для лосса игнорим токены на которых нет меток

In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset 
import tqdm

d:\condaenvs\torchenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Подготовка данных

In [2]:
PUNCT_CHARS = ".,!?;:…()"
PUNCT_MAP = {
    ".": "PERIOD",
    ",": "COMMA",
    "?": "QUESTION",
    "!": "EXCLAMATION",
    ";": "SEMICOLON",
    ":": "COLON",
    "…": "ELLIPSIS",
    "...": "ELLIPSIS",
    "?!": "QUESTION",
    "!?": "QUESTION",
    "(": "LBR",
    ")": "RBR"
}

LABEL_CASE = {"O": 0, "UPPER": 1}
LABEL_PUNCT = {"O": 0, "PERIOD": 1, "COMMA": 2, "QUESTION": 3,
               "EXCLAMATION": 4, "SEMICOLON": 5, "COLON": 6, "ELLIPSIS": 7, "LBR": 8, "RBR": 9}

In [3]:
n_train = 5000
n_val = 1000

In [4]:
dataset = load_dataset("Den4ikAI/russian_dialogues_2", split="train", streaming=True)

all_texts = [example["sample"] for example in dataset.take(n_train + n_val)]
texts = all_texts[:n_train]
val_texts = all_texts[n_train:]

In [5]:
texts[0]

['Катали на такой. Свободного ляма говоришь не было? А ежемесячно на заправки сколько бы уходило не прикидывал? ...',
 'а расход у неё какой?',
 'Написано 290л бак, запас хода 750км, ну в среднем 39л на сотню',
 'Карб от Москвича и норм :. ']

## Очистка

In [6]:
import re

def clean_text(text):
    text = re.sub(r'[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF]', '', text)
    
    text = re.sub(r'[=;:]\.', '.', text)
    text = re.sub(r'[=;:],', ',', text)
    text = re.sub(r'[=;:]\)', ')', text)
    
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

In [7]:
clean_text(" ".join(texts[0]))

'Катали на такой. Свободного ляма говоришь не было? А ежемесячно на заправки сколько бы уходило не прикидывал? ... а расход у неё какой? Написано 290л бак, запас хода 750км, ну в среднем 39л на сотню Карб от Москвича и норм .'

## Токенизация

In [8]:
WORD_RE = re.compile(r'[\w\-]+(?:\.\w+)*|[^\w\s]', re.UNICODE)

def tokenize(text: str) -> list[str]:
    return WORD_RE.findall(text)

In [9]:
test_text = "Hello world ! i Am, kek. lol :) HEHE!?"
print(tokenize(clean_text(test_text)))

['Hello', 'world', '!', 'i', 'Am', ',', 'kek', '.', 'lol', ')', 'HEHE', '!', '?']


In [10]:
def build_example(tokens: list[str]) -> dict | None:
    words, case_labels, punct_labels = [], [], []

    i = 0
    while i < len(tokens):
        token = tokens[i]
        if not re.match(r'[\w\-]', token):
            i += 1
            continue

        word = token
        punct = ""

        while i+1 < len(tokens) and tokens[i+1] in PUNCT_CHARS:
            punct += tokens[i + 1]
            i += 1

        words.append(word.lower())
        case_labels.append("UPPER" if word[0].isupper() else "O")

        if punct:
            punct_labels.append(PUNCT_MAP.get(punct, "O"))
        else:
            punct_labels.append("O")

        i += 1

    if len(words) < 3:
        return None

    return {
        "words": words,
        "case_labels": case_labels,
        "punct_labels": punct_labels,
    }

In [11]:
build_example(tokenize(clean_text(test_text)))

{'words': ['hello', 'world', 'i', 'am', 'kek', 'lol', 'hehe'],
 'case_labels': ['UPPER', 'O', 'O', 'UPPER', 'O', 'O', 'UPPER'],
 'punct_labels': ['O',
  'EXCLAMATION',
  'O',
  'COMMA',
  'PERIOD',
  'RBR',
  'QUESTION']}

In [12]:
def make_examples(texts):
    examples = []
    for row in tqdm.tqdm(texts):
        for d in row:
            cleaned = clean_text(d)
            if not cleaned:
                continue
    
            tokens = tokenize(cleaned)
            example = build_example(tokens)
            if example:
                examples.append(example)
    return examples

In [13]:
examples = make_examples(texts)
val_examples = make_examples(val_texts)

print(examples[0])
print(f"Создано примеров: {len(examples)}")

100%|██████████| 1000/1000 [00:00<00:00, 10948.44it/s]

{'words': ['катали', 'на', 'такой', 'свободного', 'ляма', 'говоришь', 'не', 'было', 'а', 'ежемесячно', 'на', 'заправки', 'сколько', 'бы', 'уходило', 'не', 'прикидывал'], 'case_labels': ['UPPER', 'O', 'O', 'UPPER', 'O', 'O', 'O', 'O', 'UPPER', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'], 'punct_labels': ['O', 'O', 'PERIOD', 'O', 'O', 'O', 'O', 'QUESTION', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']}
Создано примеров: 21592


In [14]:
def to_ids(example):
    return {
        "words": example["words"],
        "case_ids": [LABEL_CASE[c] for c in example["case_labels"]],
        "punct_ids": [LABEL_PUNCT[p] for p in example["punct_labels"]],
    }

In [15]:
ex_ids = []
for i in examples:
    ex_ids.append(to_ids(i))


In [16]:
print(ex_ids[0])

{'words': ['катали', 'на', 'такой', 'свободного', 'ляма', 'говоришь', 'не', 'было', 'а', 'ежемесячно', 'на', 'заправки', 'сколько', 'бы', 'уходило', 'не', 'прикидывал'], 'case_ids': [1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0], 'punct_ids': [0, 0, 1, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0]}


In [17]:
val_ex_ids = []
for i in val_examples:
    val_ex_ids.append(to_ids(i))

# Датасетная инженерка

In [18]:
from transformers import AutoTokenizer
from torch.utils.data import Dataset
import torch
from torch.utils.data import DataLoader

MODEL_NAME = "DeepPavlov/rubert-base-cased"
MAX_LEN = 256
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

## Выравнивание меток по subwords

In [ ]:
def align_labels(words, case_ids, punct_ids, max_len=MAX_LEN):
    encoding = tokenizer(
        words,
        is_split_into_words=True,
        truncation=True,
        max_length=max_len,
        padding=False,
    )

    word_ids = encoding.word_ids()
    seq_len = len(encoding["input_ids"])

    case_aligned = [-100] * seq_len
    punct_aligned = [-100] * seq_len

    first_seen = {}
    last_seen = {}

    for token_idx, wid in enumerate(word_ids):
        if wid is None: # Для спецтокенов
            continue
        if wid not in first_seen:
            first_seen[wid] = token_idx
        last_seen[wid] = token_idx

    for wid in first_seen:
        if wid < len(case_ids):
            case_aligned[first_seen[wid]] = case_ids[wid]
            punct_aligned[last_seen[wid]] = punct_ids[wid]

    encoding["case_labels"] = case_aligned
    encoding["punct_labels"] = punct_aligned
    return encoding

In [20]:
class PunctCaseDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        return align_labels(ex["words"], ex["case_ids"], ex["punct_ids"])

In [21]:
def collate_fn(batch):
    max_len = max(len(item["input_ids"]) for item in batch)

    input_ids = []
    attention_mask = []
    case_labels = []
    punct_labels = []

    for item in batch:
        pad_len = max_len - len(item["input_ids"])
        input_ids.append(item["input_ids"] + [tokenizer.pad_token_id] * pad_len)
        attention_mask.append(item["attention_mask"] + [0] * pad_len)
        case_labels.append(item["case_labels"] + [-100] * pad_len)
        punct_labels.append(item["punct_labels"] + [-100] * pad_len)

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        "case_labels": torch.tensor(case_labels, dtype=torch.long),
        "punct_labels": torch.tensor(punct_labels, dtype=torch.long),
    }

In [22]:
train_dataset = PunctCaseDataset(ex_ids)
train_loader = DataLoader(train_dataset, batch_size=16, collate_fn=collate_fn, shuffle=True)

val_dataset = PunctCaseDataset(val_ex_ids)
val_loader = DataLoader(val_dataset, batch_size=16, collate_fn=collate_fn, shuffle=True)

batch = next(iter(train_loader))
print("train")
print("input_ids:", batch["input_ids"].shape)
print("case_labels:", batch["case_labels"].shape)
print("punct_labels:", batch["punct_labels"].shape)

train
input_ids: torch.Size([16, 28])
case_labels: torch.Size([16, 28])
punct_labels: torch.Size([16, 28])


# Обучение

In [23]:
import torch
import torch.nn as nn
from transformers import AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW

NUM_CASE = 2
NUM_PUNCT = 10
LR = 3e-5
EPOCHS = 3

## Архитектура

In [24]:
class PunctCaseModel(nn.Module):
    def __init__(self, model_name, num_case, num_punct):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.case_head = nn.Linear(hidden, num_case)
        self.punct_head = nn.Linear(hidden, num_punct)

    def forward(self, input_ids, attention_mask):
        hidden = self.encoder(input_ids, attention_mask=attention_mask).last_hidden_state
        hidden = self.dropout(hidden)
        return self.case_head(hidden), self.punct_head(hidden)

## Обучение

In [25]:
def train(model, loader, device, epochs=EPOCHS, lr=LR):
    model.to(device)
    model.train()

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=500, num_training_steps=total_steps
    )

    ce_case = nn.CrossEntropyLoss(ignore_index=-100)
    ce_punct = nn.CrossEntropyLoss(ignore_index=-100)

    for epoch in range(epochs):
        total_loss = 0
        for i, batch in enumerate(loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            case_labels = batch["case_labels"].to(device)
            punct_labels = batch["punct_labels"].to(device)

            optimizer.zero_grad()

            case_logits, punct_logits = model(input_ids, attention_mask)

            loss_case = ce_case(
                case_logits.view(-1, NUM_CASE),
                case_labels.view(-1)
            )
            loss_punct = ce_punct(
                punct_logits.view(-1, NUM_PUNCT),
                punct_labels.view(-1)
            )

            loss = loss_case + loss_punct
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

            if (i + 1) % 100 == 0:
                print(f"Epoch {epoch+1}, step {i+1}, loss: {loss.item():.4f}")

        print(f"Epoch {epoch+1} avg loss: {total_loss / len(loader):.4f}")

In [26]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = PunctCaseModel(MODEL_NAME, NUM_CASE, NUM_PUNCT)
train(model, train_loader, device)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 20055.90it/s]
[transformers] BertModel LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1, step 100, loss: 1.0078
Epoch 1, step 200, loss: 0.8124
Epoch 1, step 300, loss: 0.6560
Epoch 1, step 400, loss: 0.5846
Epoch 1, step 500, loss: 0.8697
Epoch 1, step 600, loss: 0.6541
Epoch 1, step 700, loss: 0.4263
Epoch 1, step 800, loss: 0.7213
Epoch 1, step 900, loss: 0.5407
Epoch 1, step 1000, loss: 0.5685
Epoch 1, step 1100, loss: 0.5768
Epoch 1, step 1200, loss: 0.4744
Epoch 1, step 1300, loss: 0.5375
Epoch 1 avg loss: 0.7545
Epoch 2, step 100, loss: 0.5164
Epoch 2, step 200, loss: 0.5112
Epoch 2, step 300, loss: 0.6311
Epoch 2, step 400, loss: 0.5663
Epoch 2, step 500, loss: 0.3474
Epoch 2, step 600, loss: 0.4776
Epoch 2, step 700, loss: 0.6240
Epoch 2, step 800, loss: 0.5610
Epoch 2, step 900, loss: 0.6030
Epoch 2, step 1000, loss: 0.4669
Epoch 2, step 1100, loss: 0.5571
Epoch 2, step 1200, loss: 0.5580
Epoch 2, step 1300, loss: 0.5619
Epoch 2 avg loss: 0.5468
Epoch 3, step 100, loss: 0.4062
Epoch 3, step 200, loss: 0.6006
Epoch 3, step 300, loss: 0.5108
Epoch 3, step 

## Валидация

In [27]:
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    model.to(device)

    case_correct, case_total = 0, 0
    punct_correct, punct_total = 0, 0

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        case_labels = batch["case_labels"].to(device)
        punct_labels = batch["punct_labels"].to(device)

        case_logits, punct_logits = model(input_ids, attention_mask)

        case_pred = case_logits.argmax(dim=-1)
        punct_pred = punct_logits.argmax(dim=-1)

        mask = case_labels != -100
        case_correct += (case_pred[mask] == case_labels[mask]).sum().item()
        case_total += mask.sum().item()

        mask = punct_labels != -100
        punct_correct += (punct_pred[mask] == punct_labels[mask]).sum().item()
        punct_total += mask.sum().item()

    print(f"Case accuracy: {case_correct / case_total}")
    print(f"Punct accuracy: {punct_correct / punct_total}")

In [28]:
evaluate(model, val_loader, device)

Case accuracy: 0.9218556297440136
Punct accuracy: 0.8576741555927706


# Инференс

In [29]:
CASE_LABELS = ["O", "UPPER"]
PUNCT_LABELS = ["O", "PERIOD", "COMMA", "QUESTION", "EXCLAMATION", "SEMICOLON", "COLON", "ELLIPSIS", "LBR", "RBR"]
PUNCT_CHARS = {"PERIOD": ".", "COMMA": ",", "QUESTION": "?",
               "EXCLAMATION": "!", "SEMICOLON": ";", "COLON": ":",
               "ELLIPSIS": "...", "LBR": "(", "RBR": ")"}

@torch.no_grad()
def predict(model, words, device):
    model.eval()
    model.to(device)

    encoding = tokenizer(
        words, is_split_into_words=True,
        truncation=True, max_length=MAX_LEN,
        return_tensors="pt"
    ).to(device)

    case_logits, punct_logits = model(
        encoding["input_ids"], encoding["attention_mask"]
    )

    case_pred = case_logits.argmax(dim=-1).squeeze(0).cpu().tolist()
    punct_pred = punct_logits.argmax(dim=-1).squeeze(0).cpu().tolist()

    word_ids = encoding.word_ids(0)

    first_seen, last_seen = {}, {}
    for token_idx, wid in enumerate(word_ids):
        if wid is None:
            continue
        if wid not in first_seen:
            first_seen[wid] = token_idx
        last_seen[wid] = token_idx

    result = []
    for wid in range(len(words)):
        word = words[wid]
        case = CASE_LABELS[case_pred[first_seen[wid]]]
        punct = PUNCT_LABELS[punct_pred[last_seen[wid]]]

        if case == "UPPER":
            word = word[0].upper() + word[1:]

        punct_char = PUNCT_CHARS.get(punct, "")
        result.append(word + punct_char)

    return " ".join(result)

In [34]:
words = ["привет", "ваня", "как", "дела"]
print(predict(model, words, device))

Привет Ваня, как дела?


не идеально, но работает